# Model Context Protocol (MCP)

Model Context Protocol (MCP) is an open protocol that standardizes how applications provide tools and context to LLMs. LangChain agents can use tools defined on MCP servers using the langchain-mcp-adapters library.

**pip**

```python

pip install langchain-mcp-adapters
```

**uv**

```python

uv add langchain-mcp-adapters




In [9]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("MODEL_NAME")
MULTI_MODEL = os.getenv("MULTI_MODEL_NAME")

In [10]:
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

print("LLM loaded successfully 1✅")
message = [
    {"role": "system", "content": "You are a helpful assistant."},
]
ai_msg = llm.invoke(message)
print(ai_msg.content)

LLM loaded successfully 1✅
Hello! How can I assist you today?


In [11]:
# %pip install langchain-mcp-adapters

# import asyncio
# from langchain_mcp_adapters.client import MultiServerMCPClient  
# from langchain.agents import create_agent

# async def main():
#     client = MultiServerMCPClient(  
#         {
#             "math": {
#                 "transport": "stdio",  # Local subprocess communication
#                 "command": "python",
#                 # Absolute path to your math_server.py file
#                 "args": ["/path/to/math_server.py"],
#             },
#             "weather": {
#                 "transport": "http",  # HTTP-based remote server
#                 # Ensure you start your weather server on port 8000
#                 "url": "http://localhost:8000/mcp",
#             }
#         }
#     )

#     tools = await client.get_tools()  
#     agent = create_agent(
#         model=llm,
#         tools= tools  
#     )
#     math_response = await agent.ainvoke(
#         {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
#     )
#     weather_response = await agent.ainvoke(
#         {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]}
#     )
#     print(math_response)
#     print(weather_response)

# if __name__ == "__main__":
#     asyncio.run(main())

## Custom servers
To create a custom MCP server, use the FastMCP library:

### pip
pip install fastmcp

### uv
uv add fastmcp

## Math Server (stdio transport)

below code are working only for .py file show all deital  fastmcp server like 


In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")

### Weather server (streamable HTTP transport)

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"

if __name__ == "__main__":


    mcp.run(transport="streamable-http")

#For Run direct 

fastmcp run my_server.py:mcp

fastmcp run my_server.py:mcp --transport http --port 8000


***In which we need to create two file .py not .ipynb one server file and second is client file***


### Call Your Server
Once your server is running with HTTP transport, you can connect to it with a FastMCP client or any LLM client that supports the MCP protocol:

### Note that:

FastMCP clients are asynchronous, so we need to use asyncio.run to run the client

We must enter a client context (async with client:) before using the client

You can make multiple client calls within the same context



In [ ]:
from fastmcp import FastMCP
import random
import threading

# Create MCP server
mcp = FastMCP("SimpleTools")

@mcp.tool()
async def cricket_score(team: str) -> str:
    return f"{team} scored {random.randint(100, 350)}/{random.randint(0,10)}"

@mcp.tool()
async def stock_price(symbol: str) -> str:
    return f"{symbol.upper()} price is ${round(random.uniform(100,1500),2)}"

# Run MCP server in background
def run_server():
    mcp.run(transport="http", host="127.0.0.1", port=8000)

threading.Thread(target=run_server, daemon=True).start()

print("MCP Server Started ✅")

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.messages import AIMessage

async def setup_agent():
    client = MultiServerMCPClient(
        {
            "tools": {
                "transport": "http",
                "url": "http://127.0.0.1:8000/mcp"
            }
        }
    )

    tools = await client.get_tools()
    agent = create_agent(model=llm, tools=tools)
    return agent

agent = asyncio.run(setup_agent())
print("Agent Ready ✅")

In [ ]:
async def ask_agent(question, context=None):
    
    messages = []
    
    # optional context
    if context:
        messages.append({
            "role": "system",
            "content": context
        })
    
    messages.append({
        "role": "user",
        "content": question
    })
    
    result = await agent.ainvoke({"messages": messages})
    
    # get final AI response
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage):
            return msg.text

# Example Context
context = "You are a smart AI assistant that can use cricket and stock tools when needed."

print(asyncio.run(ask_agent("What is India cricket score?", context)))
print(asyncio.run(ask_agent("What is AAPL stock price?", context)))
print(asyncio.run(ask_agent("Tell me both in one answer.", context)))